# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use `@id` fields for precise reference. Here, we print the available record sets and their fields.

In [ ]:
# Get all record set IDs
record_sets = dataset.record_sets

print("Record Sets and their fields:")
overview = {}
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    overview[rs['@id']] = []
    # List fields
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"  Field: {f['name']} @id: {f['@id']} (type: {f.get('dataType', 'unknown')})")
            overview[rs['@id']].append(f['@id'])
    else:
        print("  No fields listed.")

# Show a sample record from each record set
for rs_id in overview.keys():
    print(f"\nSample records from {rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract record set IDs for analysis
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for {record_set_id}")

# Show columns for the main survey record set
survey_record_set_id = None
for rs in dataset.record_sets:
    if 'Survey' in rs.get('name', '') or 'survey' in rs.get('name', '').lower():
        survey_record_set_id = rs['@id']
        break
if not survey_record_set_id and all_record_set_ids:
    # Fall back to the first record set
    survey_record_set_id = all_record_set_ids[0]

print(f"\nFields (@id) for survey record set {survey_record_set_id}:")
print(dataframes[survey_record_set_id].columns.tolist())
dataframes[survey_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations are performed using field `@id`s for accuracy.

In [ ]:
# Choose a numeric field for analysis, e.g., GAD-7 score
# Suppose the GAD-7 score field has @id 'https://senscience.ai/gad7_score', and education field 'https://senscience.ai/level_of_education'
numeric_field_id = None
education_field_id = None

# Find suitable IDs from columns
for col in dataframes[survey_record_set_id].columns:
    if 'gad' in col.lower():
        numeric_field_id = col
    if 'education' in col.lower():
        education_field_id = col
if not numeric_field_id:
    numeric_field_id = dataframes[survey_record_set_id].columns[0]  # fallback
if not education_field_id:
    education_field_id = dataframes[survey_record_set_id].columns[-1]  # fallback

# Filtering by GAD-7 score (e.g., threshold = 10)
threshold = 10
filtered_df = dataframes[survey_record_set_id][dataframes[survey_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field for these filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by education level (group_field = education_field_id)
group_field = education_field_id
if group_field in dataframes[survey_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
plt.figure(figsize=(8,4))
sns.histplot(dataframes[survey_record_set_id][numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (GAD-7 score)")
plt.xlabel("GAD-7 Score")
plt.ylabel("Frequency")
plt.show()

# Boxplot of GAD-7 score by education level
plt.figure(figsize=(10,4))
sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[survey_record_set_id])
plt.title(f"{numeric_field_id} by {group_field} (Education Level)")
plt.xlabel("Education Level")
plt.ylabel("GAD-7 Score")
plt.xticks(rotation=45)
plt.show()

## 6. Conclusion
This notebook has guided you through loading, processing, and visualizing the FAIR² Mental Health Survey Dataset using `mlcroissant`. We've examined demographic and mental health scores, applied simple filtering and normalization, and visualized distributions.

- The dataset contains rich information about mental health in Kilifi County, Kenya.
- Numeric fields (such as GAD-7 scores) can be directly processed and grouped by demographics (e.g., education level).
- `mlcroissant` supports robust, AI-ready access to FAIR datasets.

**Use field and record set `@id`s for reproducible operations and deep exploration.**